# VLM Column Detection — Capability Test

Tests each VLM in two phases and saves annotated images for visual inspection.

**Phase 1 — BASELINE**: zero-shot, no hints  
**Phase 2 — ANCHOR**: 10 confirmed GT column positions injected, "find remaining"

**Image legend**
- `GREEN` = all 86 ground-truth columns
- `CYAN`  = injected seed positions (ANCHOR only)
- `RED`   = model detections this phase

**How to use**
1. Run the Setup cells once at the top
2. Run individual model cells in any order — each is independent
3. Images are saved to `/tmp/vlm_capability_test/` and displayed inline
4. Run the Summary cell at any point to see results so far

> Run this notebook from the `benchmark/` directory so `utils.py` is on the path.

---
## Setup

In [ ]:
import sys, os, json
sys.path.insert(0, ".")

from utils import (
    extract_gt_boxes, extract_bboxes, evaluate, run_inference,
    build_tile, P_BASE, P_ANCHOR,
)
from PIL import Image as _Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ANNOT_PATH = "/home/jiezhi/tmp/test_floorplan_tile.png"
CLEAN_PATH = "/tmp/test_floorplan_tile.png"
OUT_DIR    = "/tmp/vlm_capability_test"
os.makedirs(OUT_DIR, exist_ok=True)

TILE_PATH = build_tile(CLEAN_PATH, 1024)
GT        = extract_gt_boxes(ANNOT_PATH, 1024)
SEEDS     = [GT[int(i * (len(GT)-1) / 9)] for i in range(10)]

print(f"GT      : {len(GT)} columns")
print(f"Seeds   : {len(SEEDS)} spread positions")
print(f"Tile    : {TILE_PATH}")
print(f"Output  : {OUT_DIR}")

In [ ]:
# ── Annotation helper ────────────────────────────────────────────────────────
def annotate_image(dets, pool, metrics, label, out_path):
    img  = _Image.open(TILE_PATH).convert("RGB")
    draw = ImageDraw.Draw(img)
    for g in GT:
        draw.rectangle(g, outline=(0, 210, 0), width=2)
    for b in pool:
        draw.rectangle(b, outline=(0, 210, 220), width=2)
    for b in extract_bboxes(dets):
        draw.rectangle(b, outline=(230, 30, 30), width=3)
    m = metrics
    hdr = (f"{label}  |  dets={len(extract_bboxes(dets))}  "
           f"TP={m['tp']}  FP={m['fp']}  FN={m['fn']}  "
           f"P={m['p']:.2f}  R={m['r']:.2f}  F1={m['f1']:.2f}")
    draw.rectangle([0, 0, 1024, 22], fill=(0, 0, 0))
    draw.text((4, 4), hdr, fill=(255, 230, 0))
    draw.rectangle([0, 1002, 1024, 1024], fill=(20, 20, 20))
    draw.text((4, 1006), "GREEN=GT(86)   CYAN=seeds   RED=detections", fill=(180, 180, 180))
    img.save(out_path)
    return out_path


# ── Side-by-side display ─────────────────────────────────────────────────────
def show_pair(path1, path2):
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    for ax, path, title in zip(axes, [path1, path2], ["BASELINE", "ANCHOR (10 seeds)"]):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Helpers ready.")

In [ ]:
# ── Main test function ───────────────────────────────────────────────────────
results = {}   # accumulates across all model cells

def test_model(tag, timeout=300):
    safe = tag.replace(":", "_").replace(".", "_").replace("/", "_")
    print(f"\n{'='*60}\n{tag}\n{'='*60}")

    print("[1/2] BASELINE ...")
    r1 = run_inference(tag, P_BASE, TILE_PATH, timeout_s=timeout)
    m1 = evaluate(r1["dets"], GT)
    p1 = annotate_image(r1["dets"], [], m1,
                        f"{tag}  BASELINE",
                        f"{OUT_DIR}/{safe}_baseline.png")
    print(f"  dets={r1['n_dets']}  TP={m1['tp']}  FP={m1['fp']}  FN={m1['fn']}  "
          f"F1={m1['f1']:.3f}  ({r1['elapsed']}s)")
    if r1.get("error"):
        print(f"  ! error: {r1['error']}")

    print("[2/2] ANCHOR ...")
    r2 = run_inference(tag, P_ANCHOR(SEEDS), TILE_PATH, timeout_s=timeout)
    m2 = evaluate(r2["dets"], GT)
    p2 = annotate_image(r2["dets"], SEEDS, m2,
                        f"{tag}  ANCHOR(10 seeds)",
                        f"{OUT_DIR}/{safe}_anchor.png")
    print(f"  dets={r2['n_dets']}  TP={m2['tp']}  FP={m2['fp']}  FN={m2['fn']}  "
          f"F1={m2['f1']:.3f}  ({r2['elapsed']}s)")
    if r2.get("error"):
        print(f"  ! error: {r2['error']}")

    show_pair(p1, p2)

    verdict = ("HELPS" if m2['f1'] > m1['f1'] + 0.02
               else ("HURTS" if m2['f1'] < m1['f1'] - 0.02 else "same"))
    print(f"  Verdict: {verdict}")

    results[tag] = dict(
        tag=tag, baseline=m1, anchor=m2,
        t_base=r1["elapsed"], t_anchor=r2["elapsed"],
        verdict=verdict,
    )
    return results[tag]

print("test_model() ready.  results = {}")

---
## SEA-LION — Baseline Reference

`aisingapore/Gemma-SEA-LION-v4-4B-VL:latest` — ~3.3 GB  
Previous result: **P=0.254  R=0.728  F1=0.376** (best of all tested models)  
Has not been tested with memory injection yet.

In [ ]:
test_model("aisingapore/Gemma-SEA-LION-v4-4B-VL:latest", timeout=180)

---
## Small Models  (<4 GB)

In [ ]:
# moondream:latest — 1.7 GB  (fast; historically 0 TP)
test_model("moondream:latest", timeout=120)

In [ ]:
# granite3.2-vision:2b — 1.9 GB  (structured output; some spatial awareness)
test_model("granite3.2-vision:2b", timeout=120)

In [ ]:
# llava-phi3:3.8b — 2.9 GB  (prompt echo tendency; 0 TP baseline)
test_model("llava-phi3:3.8b", timeout=120)

In [ ]:
# gemma3:4b — 3.0 GB  (BEST injection candidate; consistent 10 TP on anchor)
test_model("gemma3:4b", timeout=120)

In [ ]:
# qwen2.5vl:3b — 3.2 GB  (erratic; baseline 0 but anchor sometimes fires)
test_model("qwen2.5vl:3b", timeout=300)

---
## Medium Models  (4–8 GB)

In [ ]:
# minicpm-v:8b — 5.5 GB  (good JSON; 0 TP but spatially aware)
test_model("minicpm-v:8b", timeout=300)

In [ ]:
# qwen2.5vl:7b — 6.0 GB  (larger Qwen; better reasoning than 3b)
test_model("qwen2.5vl:7b", timeout=300)

In [ ]:
# gemma4:e2b — 7.2 GB  (best JSON quality of all tested; 0 TP)
test_model("gemma4:e2b", timeout=600)

In [ ]:
# gemma3:12b — 8.1 GB  (larger gemma3; may push VRAM limits)
test_model("gemma3:12b", timeout=300)

---
## Large Models  (>8 GB)

> These require 8 GB+ free VRAM. Expect CPU offload and slow inference on 8 GB cards.

In [ ]:
# llama3.2-vision:11b — 7.9 GB  (VRAM-limited; slow via CPU offload; ~130s/phase)
test_model("llama3.2-vision:11b", timeout=420)

In [ ]:
# gemma4:e4b — 9.6 GB  (largest tested; likely needs CPU offload on 8 GB)
test_model("gemma4:e4b", timeout=600)

---
## Results Summary

Run this cell at any point to see results accumulated so far.

In [ ]:
if not results:
    print("No results yet — run a model cell first.")
else:
    print(f"\n{'Model':<48} {'BASE F1':>8} {'ANCH F1':>8} {'ANCH TP':>8}  Verdict")
    print("-" * 82)
    for tag, v in results.items():
        b  = v["baseline"]["f1"]
        a  = v["anchor"]["f1"]
        tp = v["anchor"]["tp"]
        print(f"  {tag:<46}  {b:>6.3f}  {a:>6.3f}  {tp:>6}    {v['verdict']}")
    print("-" * 82)
    print(f"  SEA-LION baseline reference: F1=0.376  (P=0.254  R=0.728)")

    # Save results JSON
    out_json = f"{OUT_DIR}/results.json"
    with open(out_json, "w") as f:
        json.dump({tag: {k: v for k, v in entry.items() if k != "tag"}
                   for tag, entry in results.items()}, f, indent=2)
    print(f"\nSaved: {out_json}")